# Блок 6 — Качество данных

Исследовательский анализ дубликатов, пропущенных значений, несогласованных значений и технических выбросов в наборе данных инцидентов SOC.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
data_path = Path("../../data/iavd_incidents_raw_target.parquet")

df = pd.read_parquet(data_path)

print("Dataset shape:", df.shape)

Dataset shape: (240295, 97)


## 6.1 Дубликаты `EntityId`

Есть ли в наборе данных дубликаты карточек инцидентов по полю `EntityId`?

In [3]:
duplicate_entity_ids = df["EntityId"].duplicated().sum()

print("Duplicate EntityId values:", duplicate_entity_ids)

Duplicate EntityId values: 0


**Вывод:** В наборе данных отсутствуют дубликаты значений `EntityId`, что означает, что каждая карточка инцидента имеет уникальный идентификатор.

## 6.2 Пустые комментарии

Сколько карточек инцидентов имеют пустое поле `Комментарий`?

In [4]:
empty_comments = df["Комментарий"].fillna("").eq("").sum()

print("Empty comments:", empty_comments)

Empty comments: 69156


**Вывод:** В 69 156 карточках инцидентов поле `Комментарий` является пустым. Это показывает, что значительная часть записей не содержит комментариев аналитиков.

## 6.3 Непустые значения в поле `Ошибки`

Сколько карточек инцидентов содержат непустое значение в поле `Ошибки`?

In [5]:
non_empty_errors = df["Ошибки"].notna().sum()

print("Non-empty Ошибки values:", non_empty_errors)

Non-empty Ошибки values: 13


**Вывод:** Только 13 карточек инцидентов содержат непустое значение в поле `Ошибки`. Следовательно, это поле почти полностью состоит из пропущенных значений и имеет ограниченную аналитическую ценность.

## 6.4 Пропущенные значения в поле `Эскалировался на L2?`

Сколько карточек инцидентов имеют пропущенное значение в поле `Эскалировался на L2?`?

In [6]:
missing_l2 = df["Эскалировался на L2?"].isna().sum()

print("Missing L2 escalation values:", missing_l2)

Missing L2 escalation values: 238539


**Вывод:** Поле `Эскалировался на L2?` содержит 238 539 пропущенных значений. Таким образом, этот признак отсутствует для подавляющего большинства карточек инцидентов.

## 6.5 Разница между NaN и `Не указан` в поле `Вердикт`

В чем разница между пропущенными значениями и явным значением `Не указан` в поле `Вердикт`?

In [7]:
verdict_nan = df["Вердикт"].isna().sum()
verdict_not_specified = (df["Вердикт"] == "Не указан").sum()

print("NaN in Вердикт:", verdict_nan)
print("Не указан in Вердикт:", verdict_not_specified)

NaN in Вердикт: 0
Не указан in Вердикт: 54870


**Вывод:** Поле `Вердикт` не содержит фактических значений NaN, однако 54 870 записей имеют явное значение `Не указан`. Это означает, что `Не указан` является отдельным категориальным значением, а не пропущенным значением.

## 6.6 Несогласованный регистр символов

Есть ли значения, которые представляют одну и ту же сущность, но записаны с использованием разного регистра символов?

In [8]:
hostnames = df["Hostname Цели"].dropna().astype(str).str.strip()

hostname_case_variants = (
    pd.DataFrame({
        "original": hostnames,
        "normalized": hostnames.str.lower()
    })
    .groupby("normalized")["original"]
    .agg(lambda x: sorted(set(x)))
)

hostname_case_variants = hostname_case_variants[
    hostname_case_variants.str.len() > 1
]

hostname_case_variants.head(20)

normalized
\ageew-aa2.iaivd.lan\ageew-aa2          [\Ageew-AA2.iaivd.lan\Ageew-AA2, \ageew-aa2.ia...
\akimova-m12.iaivd.lan\akimova-m12      [\AKIMOVA-M12.iaivd.lan\AKIMOVA-M12, \akimova-...
\akimow-i2.iaivd.lan\akimow-i2          [\AKIMOW-I2.iaivd.lan\AKIMOW-I2, \akimow-i2.ia...
\alekseew-ia.iaivd.lan\alekseew-ia      [\ALEKSEEW-IA.iaivd.lan\ALEKSEEW-IA, \alekseew...
\ananew-r2.iaivd.lan\ananew-r2          [\Ananew-R2.iaivd.lan\Ananew-R2, \ananew-r2.ia...
\arkhipow-r2.iaivd.lan\arkhipow-r2      [\ARKHIPOW-R2.iaivd.lan\ARKHIPOW-R2, \arkhipow...
\biryukow-m2.iaivd.lan\biryukow-m2      [\BIRYUKOW-M2.iaivd.lan\BIRYUKOW-M2, \biryukow...
\bogdanow-a2.iaivd.lan\bogdanow-a2      [\Bogdanow-A2.iaivd.lan\Bogdanow-A2, \bogdanow...
\borisow-d2.iaivd.lan\borisow-d2        [\BORISOW-D2.iaivd.lan\BORISOW-D2, \borisow-d2...
\burow-m2.iaivd.lan\burow-m2            [\BUROW-M2.iaivd.lan\BUROW-M2, \burow-m2.iaivd...
\dementew-e2.iaivd.lan\dementew-e2      [\Dementew-E2.iaivd.lan\Dementew-E2, \dementew...

In [9]:
print("Number of hostnames with case variants:", len(hostname_case_variants))

Number of hostnames with case variants: 108


In [10]:
crypto_columns = []

for col in df.columns:
    values = df[col].astype("string")

    if values.str.contains("cryptopro", case=False, na=False).any():
        crypto_columns.append(col)

crypto_columns

['processParentPath', 'Решение', 'CMDLINE', 'Комментарий']

In [12]:
crypto_examples = df["Комментарий"].dropna().astype(str)

crypto_examples = crypto_examples[
    crypto_examples.str.contains("cryptopro", case=False, na=False)
]

crypto_examples.head(20)

9283      Запуск Crypto Pro\\CSP\\cpanel.cpl\" - модуль ...
12668                                     Запуск  CryptoPro
39833                                      Работа CryptoPRO
39976                                      работа CryptoPro
43458                              Работа утилиты CryptoPro
43616                                      Запуск CryptoPro
43617                                    Открытие CryptoPro
46123                                     Запуск  CryptoPro
52086     проверяется какая версия CryptoPro CSP установ...
52189     происходит проверка установленной версии Crypt...
58238     активность софта Crypto Pro\Installer Cache\Cr...
96772     активность админа по установке cryptopro-5.0.1...
154369    Установка CryptoPro инженером по тех. поддержк...
159668    активность файлов программ CryptoPro_CSP_4.0.9...
160037    запуск установщика приложения CryptoPro_CSP_4....
160049    выполнение установщика программы CryptoPro_CSP...
160189                       Открытие ин

**Вывод:** В наборе данных присутствует несогласованность регистра символов в текстовых полях. Одни и те же hostname встречаются в нескольких вариантах регистра, а такие значения, как `CryptoPro`, `CryptoPRO` и `cryptopro`, также записаны по-разному. Без нормализации такие значения могут рассматриваться как разные категории при анализе или построении модели. Поэтому перед группировкой или созданием признаков текстовые поля рекомендуется привести к единому регистру.

## 6.7 Максимальное время решения и технические выбросы

Каково максимальное значение `Время решения` в часах и является ли оно реалистичным?

In [13]:
resolution_time = pd.to_numeric(
    df["Время решения"],
    errors="coerce"
)

max_resolution_hours = resolution_time.max() / 3600

print("Maximum resolution time (hours):", max_resolution_hours)

Maximum resolution time (hours): 487521350.8269445


**Вывод:** Максимальное время решения составляет приблизительно 487,5 млн часов, что явно является нереалистичным значением. Это указывает на наличие технических выбросов или поврежденных значений в поле `Время решения`. Перед анализом или построением модели такие экстремальные значения необходимо дополнительно проверить или отфильтровать.

## 6.8 Единицы измерения SLA

В каких единицах фактически хранится поле `SLA по устранению Атаки`?

In [14]:
sla_resolution = pd.to_numeric(
    df["SLA по устранению Атаки"],
    errors="coerce"
)

median_sla = sla_resolution.median()

print("Median raw SLA value:", median_sla)
print("Median SLA in hours:", median_sla / 3_600_000)

Median raw SLA value: 3600000.0
Median SLA in hours: 1.0


**Вывод:** Медианное значение `SLA по устранению Атаки` равно 3 600 000, что соответствует 1 часу при интерпретации значения в миллисекундах. Следовательно, это поле SLA хранится в миллисекундах, а не в секундах.